# 01 - Data acquisition and exploratory data analysis
**Dataset:** UCI Heart Disease, Cleveland subset (`processed.cleveland.data`, UCI id 45).  
**Goal of this notebook:** acquire the data reproducibly, validate it, clean it, and explore it with histograms, a correlation heatmap and class-balance plots. All numbers below are computed from the file, nothing is typed by hand.

> Academic exercise - not a clinically validated diagnostic tool.

In [ ]:
# Make the repository's src/ importable whether or not `pip install -e .` was run
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import os; os.chdir(ROOT)
import platform, socket, datetime
print("Run on host:", socket.gethostname(), "| Python", platform.python_version(),
      "| started", datetime.datetime.now().isoformat(timespec="seconds"))

## 1. Acquire the data (download script, checksum recorded in `data/raw/manifest.json`)

In [ ]:
import subprocess, json
subprocess.run([sys.executable, 'scripts/download_data.py'], check=True)  # raises if acquisition fails
print(json.dumps(json.load(open('data/raw/manifest.json')), indent=2))

## 2. Load the raw file and inspect it
The file has no header; `?` marks missing values.

In [ ]:
from heartrisk.data import load_raw, clean
from heartrisk.schema import FEATURES, NUMERIC_FEATURES, BINARY_FEATURES, CATEGORICAL_FEATURES
raw = load_raw()
print(raw.shape)
raw.head()

In [ ]:
raw.info()

In [ ]:
raw.describe().T.round(2)

## 3. Data quality: missing values, duplicates, invalid codes

In [ ]:
print('Missing values per column:')
print(raw.isna().sum()[raw.isna().sum() > 0])
print('Rows with any missing value:', int(raw.isna().any(axis=1).sum()))
print('Exact duplicate rows:', int(raw.duplicated().sum()))
from heartrisk.data import invalid_value_mask
print('Rows with out-of-range values / undefined codes:', int(invalid_value_mask(raw).sum()))

## 4. Clean
* `?` -> missing (kept as NaN; **imputed inside the model pipeline**, fitted on training folds only, to avoid leakage)
* exact duplicates removed, rows with impossible values removed
* target: original `num` (0-4) -> binary `target` (0 = absent, 1 = present)

In [ ]:
df, report = clean(raw)
import json; print(json.dumps(report.to_dict(), indent=2))
df.to_csv('data/processed/heart_clean.csv', index=False)
print('saved data/processed/heart_clean.csv', df.shape)

### Feature types used by the pipeline

In [ ]:
import pandas as pd
pd.DataFrame({'feature': NUMERIC_FEATURES + BINARY_FEATURES + CATEGORICAL_FEATURES,
              'type': ['numeric (median impute + standard scale)']*len(NUMERIC_FEATURES)
                    + ['binary (mode impute)']*len(BINARY_FEATURES)
                    + ['categorical (mode impute + one-hot)']*len(CATEGORICAL_FEATURES)})

## 5. Visual EDA
Figures are written to `reports/figures/eda/` and used in the report.

In [ ]:
from heartrisk import eda
summary = eda.run()
from IPython.display import Image, display
for f in ['raw_target_num', 'class_balance', 'missing_values']:
    display(Image(f'reports/figures/eda/{f}.png'))

### Histograms of numeric features by class

In [ ]:
display(Image('reports/figures/eda/numeric_histograms.png'))

### Disease rate by categorical / binary feature

In [ ]:
display(Image('reports/figures/eda/categorical_disease_rate.png'))

### Correlation heatmap

In [ ]:
display(Image('reports/figures/eda/correlation_heatmap.png'))

## 6. Observations (generated from the data)

In [ ]:
t = summary['target_counts']; n = summary['rows_clean']
print(f"Clean dataset: {n} patients, {summary['n_features']} features.")
print(f"Class balance: {t['0']} without disease ({t['0']/n:.1%}), {t['1']} with disease ({t['1']/n:.1%}) "
      f"-> {'roughly balanced; accuracy is meaningful but ROC-AUC/recall are also reported' if 0.4 <= t['1']/n <= 0.6 else 'imbalanced; rely on ROC-AUC, precision and recall'}.")
print("Missing values in raw file:", summary['missing_raw'])
print("\nStrongest linear associations with the target (Pearson r):")
for k, v in list(summary['corr_with_target_sorted'].items())[:6]:
    print(f"  {k:9s} {v:+.3f}")
print("\nMean of numeric features by class:")
print(pd.DataFrame(summary['numeric_mean_by_class']).T)

**Leakage check:** the only target-derived column is `num`, which is converted into `target` and then dropped; no other feature is computed from the diagnosis. Scaling, imputation and encoding are fitted inside the scikit-learn pipeline during cross-validation.